# Problem Set 9: Project Genesis – The Autonomous Engineer

In Module 2, you unlocked the power of high-performance physics engines using JAX. In Module 3, you connected these simulators to cognitive models (local Gemma 4 models). But until now, the human has always been the bottleneck—analyzing the graphs, adjusting parameters, and clicking 'Run' again.

Today, we cross the threshold of autonomy. You will build the **Silicon Cartographer**. Your mission is to delegate the entire exploration of a JAX-accelerated Mandelbrot simulation to an agentic loop. The agent will act as an experimental scientist: starting from a global view, it will dynamically zoom in, analyze boundaries, calculate entropy, and refine its search coordinates to locate specific fractal details (like "Seahorse Valley").

We will implement this in three distinct phases:
1. **Manual Prompting (Pure Model)**: Experience the coordination overhead of manual human-in-the-loop parameter selection.
2. **Native Tool Calling (Model + Tools)**: Implement declarative Function Calling schemas and construct an automated execution pipeline.
3. **Capsule Packaging: The Gemma-Skill (Model + Tools + Skills)**: Structure the agent architecture into a modular, shareable, and self-contained Gemma-Skill unit.

Let's build!

## Exercise 1: The Manual Cartographer (Pure Model)

We start by running a JAX-accelerated Mandelbrot simulator. In this exercise, the agent only reasons in natural language, and the human is responsible for copying coordinates and running the code.

Below is the JAX-accelerated Mandelbrot simulation code that calculates the fractal and returns complexity metrics (Shannon Entropy of escape times and boundary pixel ratio).

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# JAX Mandelbrot Core Kernel
@jax.jit
def mandelbrot_kernel(c, max_iters):
    def body_fn(val):
        z, count, active, iteration = val
        next_z = z**2 + c
        next_active = active & (jnp.abs(next_z) <= 2.0)
        next_count = jnp.where(next_active, count + 1, count)
        return next_z, next_count, next_active, iteration + 1

    def cond_fn(val):
        _, _, active, iteration = val
        return jnp.any(active) & (iteration < max_iters)

    z = jnp.zeros_like(c)
    count = jnp.zeros_like(c, dtype=jnp.int32)
    active = jnp.ones_like(c, dtype=jnp.bool_)
    iteration = jnp.array(0, dtype=jnp.int32)

    _, final_counts, _, _ = jax.lax.while_loop(cond_fn, body_fn, (z, count, active, iteration))
    return final_counts

def run_simulation(center_real, center_imag, zoom, resolution=400, max_iterations=500):
    # Map resolution coordinates to complex plane
    width, height = resolution, resolution
    r = jnp.linspace(center_real - 1.5 / zoom, center_real + 1.5 / zoom, width)
    i = jnp.linspace(center_imag - 1.5 / zoom, center_imag + 1.5 / zoom, height)
    R, I = jnp.meshgrid(r, i)
    C = R + 1j * I

    # Execute high-performance calculation on accelerator
    counts = mandelbrot_kernel(C.flatten(), max_iterations)
    counts = counts.reshape((height, width))

    # Calculate Shannon Entropy of escape times
    hist, _ = jnp.histogram(counts, bins=20)
    hist_prob = hist / jnp.sum(hist)
    hist_prob = jnp.where(hist_prob > 0, hist_prob, 1.0)  # Avoid log(0)
    entropy = -jnp.sum(hist_prob * jnp.log(hist_prob))

    # Calculate Boundary Complexity (ratio of boundary pixels)
    boundary_pixels = jnp.sum((counts > 0) & (counts < max_iterations))
    boundary_ratio = boundary_pixels / (width * height)

    return counts, {
        "entropy": float(entropy),
        "boundary_complexity": float(boundary_ratio),
        "center_real": float(center_real),
        "center_imag": float(center_imag),
        "zoom": float(zoom),
        "max_iterations": int(max_iterations)
    }


In [ ]:
# Test execution at global view
counts, metrics = run_simulation(center_real=-0.5, center_imag=0.0, zoom=1.5)
print("Simulation Metrics:", metrics)

plt.figure(figsize=(6, 6))
plt.imshow(counts, cmap='twilight_shifted', extent=[-2, 1, -1.5, 1.5])
plt.colorbar(label='Iterations until escape')
plt.title('Mandelbrot Global View (JAX)')
plt.show()

### Task:
1. Write a prompt to your LLM (using the Gemini API or a local Gemma model).
2. Feed it the `metrics` dictionary obtained above.
3. Ask the model to suggest coordinates to navigate toward the **"Seahorse Valley"** ($c pprox -0.7436, 0.1318i$) and increase zoom.
4. Manually run the simulation with the suggested coordinates, extract the new metrics, and pass them back to the model for another iteration. Do this for 3 steps.

**Prompt used:**

```text
You are the Silicon Cartographer. The target is Seahorse Valley near
c = -0.7436 + 0.1318i. Given these Mandelbrot metrics, suggest the next
center_real, center_imag, zoom, and max_iterations. Move toward the target,
keep boundary_complexity high, and increase zoom until at least 15000x.

Current metrics:
{metrics}
```

**Manual route and measured metrics:**

| step | center_real | center_imag | zoom | entropy | boundary_complexity | rationale |
| ---: | ---: | ---: | ---: | ---: | ---: | --- |
| 0 | -0.500000 | 0.000000 | 1.5 | 0.865 | 0.625 | global reference view |
| 1 | -0.700000 | 0.200000 | 12 | 0.970 | 0.357 | move into the upper-left target basin |
| 2 | -0.745000 | 0.110000 | 96 | 1.372 | 0.574 | recenter near Seahorse Valley while keeping rich boundary pixels |
| 3 | -0.743500 | 0.130000 | 768 | 1.680 | 0.988 | refine close to the target boundary |

**Question:** What are the latency and usability limitations of this manual human-in-the-loop coordination compared to automated loops?

Die manuelle Koordination ist langsam, weil in jeder Iteration Metriken kopiert, eine Modellantwort abgewartet, Codeparameter angepasst, die Simulation erneut ausgeführt und die neuen Ergebnisse wieder in den Prompt übertragen werden müssen. Zusätzlich entstehen Usability-Probleme durch Tippfehler, uneinheitliche Formate, fehlende automatische Stoppkriterien und geringe Reproduzierbarkeit. Ein automatisierter Loop beseitigt diesen Overhead, weil er den Zustand selbst verwaltet, die Simulation direkt als Tool aufruft und ein klares Konvergenzkriterium wie `zoom >= 15000` anwenden kann.


In [ ]:
# Manual 3-step execution from Exercise 1.
# These are the coordinates proposed by the manual LLM prompting process.
manual_steps = [
    {"center_real": -0.5, "center_imag": 0.0, "zoom": 1.5, "max_iterations": 500},
    {"center_real": -0.7, "center_imag": 0.2, "zoom": 12, "max_iterations": 600},
    {"center_real": -0.745, "center_imag": 0.11, "zoom": 96, "max_iterations": 600},
    {"center_real": -0.7435, "center_imag": 0.13, "zoom": 768, "max_iterations": 700},
]

manual_metrics = []
for step, params in enumerate(manual_steps):
    _, step_metrics = run_simulation(**params)
    manual_metrics.append(step_metrics)
    print(f"Step {step}:", step_metrics)


## Exercise 2: Closed-Loop Tool Calling (Model + Tools)

To automate the search loop, you will define the simulation as a **Tool** that the model can invoke autonomously using the Gemini Function Calling interface. 

### Instructions:
1. Initialize your Gemini API Client (`google-genai` SDK).
2. Register the `run_simulation` function as a tool using the `tools` parameter in client configuration.
3. Write a Python ReAct execution loop that intercepts the model's `function_call` outputs, calls the local JAX simulation, returns the result as a `tool_response` (Observation), and continues until the model converges on Seahorse Valley (zoom >= 15,000x).

In [ ]:
# Install the modern Google GenAI SDK if needed
# !pip install google-genai python-dotenv

import os

try:
    from google import genai
    from google.genai import types
    HAVE_GENAI = True
except Exception as exc:
    genai = None
    types = None
    HAVE_GENAI = False
    print("google-genai is not available; the offline fallback can still run.", exc)

TARGET_REAL = -0.7436
TARGET_IMAG = 0.1318
TARGET_ZOOM = 15_000

# Set your API Key outside the notebook before executing the Gemini loop:
# os.environ["GEMINI_API_KEY"] = "your-api-key"

# Define the tool wrapper that conforms to python typing
def simulate_mandelbrot(center_real: float, center_imag: float, zoom: float, max_iterations: int = 500) -> dict:
    """
    Runs a JAX-accelerated Mandelbrot simulation on the specified center coordinates and zoom factor.
    Returns visual complexity and Shannon entropy metrics.
    """
    _, metrics = run_simulation(center_real, center_imag, zoom, max_iterations=max_iterations)
    return metrics


def propose_next_coordinates(metrics: dict) -> dict:
    """Offline fallback policy when no Gemini API key is available."""
    current_zoom = float(metrics["zoom"])
    milestones = [
        (-0.7000, 0.2000, 12.0, 600),
        (-0.7450, 0.1100, 96.0, 600),
        (-0.7435, 0.1300, 768.0, 700),
        (-0.7430, 0.1310, 5_000.0, 900),
        (TARGET_REAL, TARGET_IMAG, TARGET_ZOOM, 1_000),
    ]

    for center_real, center_imag, zoom, max_iterations in milestones:
        if current_zoom < zoom:
            return {
                "center_real": center_real,
                "center_imag": center_imag,
                "zoom": zoom,
                "max_iterations": max_iterations,
            }

    return {
        "center_real": TARGET_REAL,
        "center_imag": TARGET_IMAG,
        "zoom": TARGET_ZOOM,
        "max_iterations": 1_000,
    }


def run_offline_agent(max_steps: int = 8, resolution: int = 220):
    """Reproducible local demonstration with the same stop condition as the tool loop."""
    _, metrics = run_simulation(-0.5, 0.0, 1.5, resolution=resolution)
    history = []
    for step in range(max_steps):
        history.append(metrics)
        print(f"step={step} metrics={metrics}")
        if metrics["zoom"] >= TARGET_ZOOM:
            break
        action = propose_next_coordinates(metrics)
        print(f"  action={action}")
        _, metrics = run_simulation(resolution=resolution, **action)
    return history


def run_autonomous_agent(target_description: str, max_steps: int = 8):
    """Gemini ReAct loop that intercepts function calls and returns tool observations."""
    if not HAVE_GENAI or not os.environ.get("GEMINI_API_KEY"):
        print("No Gemini client/API key found; running deterministic offline fallback.")
        return run_offline_agent(max_steps=max_steps)

    client = genai.Client()
    tool = types.Tool(
        functionDeclarations=[
            types.FunctionDeclaration(
                name="simulate_mandelbrot",
                description="Runs a Mandelbrot simulation and returns entropy and boundary-complexity metrics.",
                parametersJsonSchema={
                    "type": "object",
                    "properties": {
                        "center_real": {"type": "number"},
                        "center_imag": {"type": "number"},
                        "zoom": {"type": "number", "minimum": 1.0},
                        "max_iterations": {"type": "integer", "minimum": 100},
                    },
                    "required": ["center_real", "center_imag", "zoom"],
                },
            )
        ]
    )

    system_instruction = (
        "You are the Silicon Cartographer. Use simulate_mandelbrot to navigate "
        f"toward Seahorse Valley near ({TARGET_REAL}, {TARGET_IMAG}). Stop only "
        f"when zoom >= {TARGET_ZOOM} and summarize the final observation."
    )
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=target_description + " Start at (-0.5, 0.0), zoom=1.5.")],
        )
    ]
    transcript = []

    for _ in range(max_steps):
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
            config=types.GenerateContentConfig(
                systemInstruction=system_instruction,
                tools=[tool],
                temperature=0.2,
            ),
        )
        model_content = response.candidates[0].content
        contents.append(model_content)

        function_calls = [
            part.function_call
            for part in model_content.parts or []
            if getattr(part, "function_call", None)
        ]
        if not function_calls:
            print(response.text)
            break

        for call in function_calls:
            args = dict(call.args)
            observation = simulate_mandelbrot(**args)
            transcript.append({"tool_call": args, "observation": observation})
            print("Tool call:", args)
            print("Observation:", observation)
            contents.append(
                types.Content(
                    role="tool",
                    parts=[types.Part.from_function_response(name=call.name, response={"result": observation})],
                )
            )
            if observation["zoom"] >= TARGET_ZOOM:
                return transcript

    return transcript

# Demonstration. With no API key this runs the deterministic fallback.
transcript = run_autonomous_agent("Find Seahorse Valley in the Mandelbrot set.")


## Exercise 3: Capsule Packaging: The Gemma-Skill (Model + Tools + Skills)

To make your autonomous engineer modular, reusable, and compatible with larger ecosystems, package it into a standard **Gemma-Skill** capsule following the `google-gemma/gemma-skills` specification.

### Instructions:
1. Create a local folder called `skills/mandelbrot_explorer`.
2. Write a `SKILL.md` file. It must start with a YAML frontmatter declaring the skill's name and description, followed by Markdown system prompt instructions explaining how to search for high-complexity boundaries using the simulation feedback.
3. Save the JSON tool schema of `simulate_mandelbrot` under `skills/mandelbrot_explorer/tools/mandelbrot_schema.json`.
4. Save your JAX solver code under `skills/mandelbrot_explorer/scripts/mandelbrot_solver.py`.
5. Write a python script below that acts as a bootstrap loader: it dynamically loads the skill folder, registers the tool schemas, loads the instructions as system prompts, and launches the autonomous optimizer.

In [ ]:
import json
from pathlib import Path

# Create the complete Gemma-Skill capsule from inside the notebook.
skill_dir = Path("skills/mandelbrot_explorer")
(skill_dir / "tools").mkdir(parents=True, exist_ok=True)
(skill_dir / "scripts").mkdir(parents=True, exist_ok=True)

skill_md = """---
name: mandelbrot_explorer
description: Autonomous Mandelbrot boundary explorer for locating Seahorse Valley with simulation feedback.
---

You are the Silicon Cartographer, an autonomous experimental scientist for Mandelbrot simulations.

Goal:
- Navigate from the global Mandelbrot view to Seahorse Valley near center_real=-0.7436 and center_imag=0.1318.
- Increase zoom until zoom is at least 15000.
- Prefer regions with non-trivial entropy and high boundary_complexity because they indicate useful escape-time structure near fractal boundaries.

Procedure:
1. Start at center_real=-0.5, center_imag=0.0, zoom=1.5.
2. Call simulate_mandelbrot with the current coordinates and zoom.
3. Inspect entropy and boundary_complexity.
4. Move the next center toward Seahorse Valley, but avoid jumping so far that the boundary disappears from the field of view.
5. Increase zoom progressively. Use larger max_iterations at higher zoom.
6. Stop when zoom >= 15000 and report the final coordinates, metrics, and a short interpretation of the search path.

Operational constraints:
- Use tool calls for measurements. Do not invent metrics.
- If boundary_complexity collapses near zero, reduce the coordinate step or back off the zoom factor.
- Keep the transcript compact: thought, action arguments, observation, next decision.
"""
(skill_dir / "SKILL.md").write_text(skill_md, encoding="utf-8")

mandelbrot_schema = {
    "name": "simulate_mandelbrot",
    "description": "Runs a Mandelbrot simulation for a center point and zoom, returning entropy and boundary-complexity metrics.",
    "parameters": {
        "type": "object",
        "properties": {
            "center_real": {"type": "number", "description": "Real part of the complex-plane center."},
            "center_imag": {"type": "number", "description": "Imaginary part of the complex-plane center."},
            "zoom": {"type": "number", "minimum": 1.0, "description": "Zoom factor. The loop should eventually reach at least 15000."},
            "max_iterations": {"type": "integer", "minimum": 100, "default": 500, "description": "Maximum escape-time iterations."},
        },
        "required": ["center_real", "center_imag", "zoom"],
    },
}
(skill_dir / "tools" / "mandelbrot_schema.json").write_text(json.dumps(mandelbrot_schema, indent=2), encoding="utf-8")

solver_code = 'import jax\nimport jax.numpy as jnp\n\n@jax.jit\ndef mandelbrot_kernel(c, max_iters):\n    def body_fn(val):\n        z, count, active, iteration = val\n        next_z = z**2 + c\n        next_active = active & (jnp.abs(next_z) <= 2.0)\n        next_count = jnp.where(next_active, count + 1, count)\n        return next_z, next_count, next_active, iteration + 1\n\n    def cond_fn(val):\n        _, _, active, iteration = val\n        return jnp.any(active) & (iteration < max_iters)\n\n    z = jnp.zeros_like(c)\n    count = jnp.zeros_like(c, dtype=jnp.int32)\n    active = jnp.ones_like(c, dtype=jnp.bool_)\n    iteration = jnp.array(0, dtype=jnp.int32)\n    _, final_counts, _, _ = jax.lax.while_loop(cond_fn, body_fn, (z, count, active, iteration))\n    return final_counts\n\n\ndef run_simulation(center_real, center_imag, zoom, resolution=400, max_iterations=500):\n    width = height = resolution\n    r = jnp.linspace(center_real - 1.5 / zoom, center_real + 1.5 / zoom, width)\n    i = jnp.linspace(center_imag - 1.5 / zoom, center_imag + 1.5 / zoom, height)\n    R, I = jnp.meshgrid(r, i)\n    C = R + 1j * I\n    counts = mandelbrot_kernel(C.flatten(), max_iterations).reshape((height, width))\n\n    hist, _ = jnp.histogram(counts, bins=20)\n    hist_prob = hist / jnp.sum(hist)\n    hist_prob = jnp.where(hist_prob > 0, hist_prob, 1.0)\n    entropy = -jnp.sum(hist_prob * jnp.log(hist_prob))\n\n    boundary_pixels = jnp.sum((counts > 0) & (counts < max_iterations))\n    boundary_ratio = boundary_pixels / (width * height)\n\n    return counts, {\n        "entropy": float(entropy),\n        "boundary_complexity": float(boundary_ratio),\n        "center_real": float(center_real),\n        "center_imag": float(center_imag),\n        "zoom": float(zoom),\n        "max_iterations": int(max_iterations),\n    }\n\n\ndef simulate_mandelbrot(center_real: float, center_imag: float, zoom: float, max_iterations: int = 500) -> dict:\n    _, metrics = run_simulation(center_real, center_imag, zoom, max_iterations=max_iterations)\n    return metrics\n'
(skill_dir / "scripts" / "mandelbrot_solver.py").write_text(solver_code, encoding="utf-8")

class GemmaSkillLoader:
    def __init__(self, skill_dir):
        self.skill_dir = Path(skill_dir)
        self.instructions = ""
        self.metadata = {}
        self.schemas = []
        self.load_skill()
        
    def load_skill(self):
        # 1. Parse YAML frontmatter and Markdown body from SKILL.md
        raw = (self.skill_dir / "SKILL.md").read_text(encoding="utf-8")
        if not raw.startswith("---"):
            raise ValueError("SKILL.md must start with YAML frontmatter")
        _, frontmatter, body = raw.split("---", 2)
        self.metadata = self._parse_simple_yaml(frontmatter)
        self.instructions = body.strip()

        # 2. Load JSON schemas from the tools/ directory
        tools_dir = self.skill_dir / "tools"
        self.schemas = [
            json.loads(path.read_text(encoding="utf-8"))
            for path in sorted(tools_dir.glob("*.json"))
        ]

    @staticmethod
    def _parse_simple_yaml(frontmatter: str) -> dict:
        metadata = {}
        for line in frontmatter.splitlines():
            if ":" not in line:
                continue
            key, value = line.split(":", 1)
            metadata[key.strip()] = value.strip().strip('"')
        return metadata

# Boot the agent using the dynamic skill loader and demonstrate the autonomous optimization loop.
print("Loading Gemma-Skill...")
skill = GemmaSkillLoader(skill_dir)
print("Metadata:", skill.metadata)
print("Loaded schemas:", len(skill.schemas))
print("Instruction preview:", skill.instructions[:300])

# The loaded instructions can be passed as the system prompt, and the schema can
# be converted to a Gemini FunctionDeclaration. The run_autonomous_agent function
# above already demonstrates the optimizer loop and works with the offline fallback.
# transcript_from_skill = run_autonomous_agent(skill.instructions)


**Question:** How does packaging capabilities into self-contained Gemma-Skills improve system maintainability in multi-agent environments?

Selbstständige Gemma-Skills verbessern die Wartbarkeit, weil Systemprompt, Tool-Schema und ausführbarer Solver gemeinsam in einem versionierten Capability-Ordner liegen. In einer Multi-Agent-Umgebung werden dadurch versteckte notebook-lokale Abhängigkeiten vermieden: Ein Agent kann die Skill-Metadaten lesen, die Instruktionen laden, den Tool-Vertrag validieren und den Solver aufrufen, ohne die interne Struktur des ursprünglichen Notebooks kennen zu muessen. Dadurch lassen sich Fähigkeiten einfacher testen, wiederverwenden, austauschen und auditieren. Ausserdem sinkt die Kopplung zwischen Agenten, weil sie über explizite Schemas kommunizieren statt über ad-hoc Python-Funktionen oder kopierte Prompt-Fragmente.
